# Facade Design Pattern 

explained using the Home Theater example.

#### The Concept

The Facade Pattern provides a simplified interface to a complex library or a set of classes. 

Analogy: A `"Master Button"` on a remote.
- **Without Facade**: You have to turn on the TV, switch the Input to HDMI, turn on the Amplifier, set the volume to 20, dim the lights, and close the blinds manually.
- **With Facade**: You press one button: "Movie Mode". The Facade handles all the complexity behind the scenes.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we explicitly define all the subsystems (Lights, Projector, Sound). The Facade takes these objects in its constructor (Dependency Injection) and orchestrates them.

#### THE COMPLEX SUBSYSTEMS

In [1]:
class Amplifier:
    def on(self): print("Amp: Turning ON")
    def set_volume(self, level): print(f"Amp: Volume set to {level}")

class Projector:
    def on(self): print("Projector: Turning ON")
    def set_input(self, source): print(f"Projector: Input set to {source}")

class Lights:
    def dim(self, level): print(f"Lights: Dimming to {level}%")

class Screen:
    def down(self): print("Screen: Lowering...")

#### THE FACADE

In [2]:
class HomeTheaterFacade:
    def __init__(self, amp, projector, lights, screen):
        # We manually inject every single dependency
        self.amp = amp
        self.projector = projector
        self.lights = lights
        self.screen = screen

    def watch_movie(self, movie_name):
        print(f"\n--- Get ready to watch '{movie_name}' ---")
        self.lights.dim(10)
        self.screen.down()
        self.projector.on()
        self.projector.set_input("DVD")
        self.amp.on()
        self.amp.set_volume(5)

#### CLIENT CODE

In [3]:
def main():
    # The client has to know about ALL the subsystems to set up the Facade
    amp = Amplifier()
    proj = Projector()
    lights = Lights()
    screen = Screen()

    # Create Facade
    home_theater = HomeTheaterFacade(amp, proj, lights, screen)

    # Use Facade
    home_theater.watch_movie("Inception")

if __name__ == "__main__":
    main()


--- Get ready to watch 'Inception' ---
Lights: Dimming to 10%
Screen: Lowering...
Projector: Turning ON
Projector: Input set to DVD
Amp: Turning ON
Amp: Volume set to 5


## The Pythonic Way

In Python, we prioritize **Simplicity** and **"Batteries Included"**.
- **Encapsulation**: The Facade should probably initialize its own subsystems if they are standard, so the client line is shorter.
- **Functions over Classes**: Sometimes, a Facade doesn't even need to be a class. A simple high-level function in a module is the most Pythonic facade.

Here is a cleaner Class-based approach where the Facade manages the messy setup for you.

#### THE SUBSYSTEMS (Same as before)

In [4]:
class Amplifier:
    def on(self): print("Amp: ON")
    def set_volume(self, lvl): print(f"Amp: Volume {lvl}")

class Projector:
    def on(self): print("Projector: ON")
    def tv_mode(self): print("Projector: Mode TV")

class Lights:
    def dim(self): print("Lights: Dimmed")

#### THE PYTHONIC FACADE

In [5]:
class SmartHome:
    """
    The Facade initializes the subsystems internally.
    The client doesn't need to know 'Amplifier' or 'Lights' even exist.
    """
    def __init__(self):
        self.amp = Amplifier()
        self.proj = Projector()
        self.lights = Lights()

    def movie_mode(self):
        print("\n--- 🍿 Starting Movie Mode 🍿 ---")
        self.lights.dim()
        self.proj.on()
        self.proj.tv_mode()
        self.amp.on()
        self.amp.set_volume(15)

    def gaming_mode(self):
        print("\n--- 🎮 Starting Gaming Mode 🎮 ---")
        self.lights.dim()
        self.proj.on()
        self.amp.on()
        self.amp.set_volume(30)

#### CLIENT CODE

In [6]:
def main():
    # Look how clean the client code is now:
    home = SmartHome()
    
    home.movie_mode()
    home.gaming_mode()

if __name__ == "__main__":
    main()


--- 🍿 Starting Movie Mode 🍿 ---
Lights: Dimmed
Projector: ON
Projector: Mode TV
Amp: ON
Amp: Volume 15

--- 🎮 Starting Gaming Mode 🎮 ---
Lights: Dimmed
Projector: ON
Amp: ON
Amp: Volume 30


The Ultimate Pythonic Facade: `__init__.py` </br>
In real Python projects (libraries), the Facade pattern is often implemented at the **Module Level.**

Imagine a complex package folder structure:
```
my_library/
    ├── database/
    │   └── connection.py  (Class DatabaseConnection)
    ├── processing/
    │   └── math.py        (Class ComplexMath)
    └── __init__.py        <-- THE FACADE
```

Inside `my_library/__init__.py`:
```py
from .database.connection import DatabaseConnection
from .processing.math import ComplexMath

def easy_run():
    # This function is a Facade!
    db = DatabaseConnection()
    calc = ComplexMath()
    db.connect()
    return calc.process(db.data)
```

User code:
```py
import my_library

# The user uses the package facade directly
result = my_library.easy_run()
```

#### Summary

- **Java Way**: Explicitly inject dependencies. Good for testing, bad for brevity.
- **Python Way**: Hide the initialization mess inside the Facade. Or better yet, use a simple Function or Module as your Facade.

# Facade Design Pattern 

explained using a common software industry scenario: An E-Commerce Order Processing System.

#### The Concept

The Facade Pattern provides a simplified "Front Door" interface to a complex system of classes, libraries, or frameworks. **The Problem**: Placing an order involves interacting with 4 different microservices: `Inventory`, `Payment`, `Shipping`, and `Notification`. If the Frontend (Client) has to call all 4 separately, the code becomes messy, tightly coupled, and hard to test. 

**The Solution**: Create a specific `OrderFacade` class. The Client just calls `orderFacade.place_order()`, and the Facade handles the complex coordination behind the scenes.

**Analogy**: A Car Dashboard. You press the "Start Button" (Facade). You don't manually inject fuel, spark the plugs, and engage the starter motor (Subsystems).

## The Classic OOP Way (Java-Style)

In strict OOP, we create distinct classes for every subsystem. The Facade wraps them and manages the sequence of execution.

#### THE COMPLEX SUBSYSTEMS

In [7]:
class InventorySystem:
    def check_stock(self, item_id: str) -> bool:
        print(f"   📦 [Inventory] Checking stock for {item_id}...")
        return True

    def reserve_item(self, item_id: str):
        print(f"   📦 [Inventory] Item {item_id} reserved.")

class PaymentGateway:
    def process_payment(self, amount: float) -> bool:
        print(f"   💳 [Payment] Charging ${amount}...")
        return True

class ShippingService:
    def calculate_shipping(self, zip_code: str) -> float:
        return 15.0

    def generate_label(self, address: str) -> str:
        print(f"   🚚 [Shipping] Label created for {address}.")
        return "TRACKING-123"

class EmailService:
    def send_confirmation(self, email: str):
        print(f"   📧 [Email] Confirmation sent to {email}.")

#### THE FACADE

In [8]:
class OrderFacade:
    def __init__(self):
        # The Facade owns the instances of the subsystems
        self.inventory = InventorySystem()
        self.payment = PaymentGateway()
        self.shipping = ShippingService()
        self.email = EmailService()

    def place_order(self, item_id: str, amount: float, address: str, email: str):
        print("--- Order Facade: Starting Process ---")
        
        # 1. Check Inventory
        if not self.inventory.check_stock(item_id):
            print("❌ Error: Out of stock.")
            return

        # 2. Process Payment
        if not self.payment.process_payment(amount):
            print("❌ Error: Payment failed.")
            return

        # 3. Handle Logistics
        self.inventory.reserve_item(item_id)
        self.shipping.generate_label(address)
        
        # 4. Notify
        self.email.send_confirmation(email)
        
        print("✅ Order Facade: Process Complete.")

#### CLIENT CODE

In [9]:
def main():
    # The client doesn't need to import or know about Inventory, Payment, etc.
    # It just needs the Facade.
    facade = OrderFacade()
    
    facade.place_order("Laptop-X", 1200.00, "123 Tech St", "user@example.com")

if __name__ == "__main__":
    main()

--- Order Facade: Starting Process ---
   📦 [Inventory] Checking stock for Laptop-X...
   💳 [Payment] Charging $1200.0...
   📦 [Inventory] Item Laptop-X reserved.
   🚚 [Shipping] Label created for 123 Tech St.
   📧 [Email] Confirmation sent to user@example.com.
✅ Order Facade: Process Complete.


## The Pythonic Way (Module Facade / init.py)

In Python, we often don't need a specific `Facade` class. The Python Module System itself is a natural Facade. By using an `__init__`.py file or a simple high-level function, we can hide the internal implementation details of a package.

In this example, we treat the `place_order` function as the Facade. We also assume the subsystems might be complex functions rather than classes (functional approach).

#### THE SUBSYSTEMS (Internal Logic)

In [10]:
# Imagine these are complex libraries imported from other files
def _check_inventory(sku: str) -> bool:
    print(f"📦 Checking Stock: {sku}")
    return True

def _charge_card(token: str, amt: int):
    print(f"💳 Charged ${amt} to card {token}")

def _ship_package(sku: str, addr: str):
    print(f"🚚 Shipped {sku} to {addr}")

#### THE PYTHONIC FACADE (A High-Level Function)

In [11]:
# By using an underscore in subsystem names (e.g., _check_inventory),
# we imply they are internal. This function is the only public API.

def buy_product(sku: str, price: int, user_info: dict):
    """
    This function acts as the Facade. 
    It creates a simple interface for a complex multi-step workflow.
    """
    print(f"--- Buying {sku} ---")
    
    try:
        # Step 1: Inventory
        if not _check_inventory(sku):
            raise Exception("Out of Stock")
            
        # Step 2: Payment
        _charge_card(user_info['card'], price)
        
        # Step 3: Shipping
        _ship_package(sku, user_info['address'])
        
        print("✅ Purchase successful!")
        
    except Exception as e:
        print(f"❌ Purchase failed: {e}")

#### CLIENT CODE

In [12]:
def main():
    user = {
        "card": "VISA-4242",
        "address": "NY, USA"
    }

    # The client just calls one function.
    # No need to instantiate 4 different classes.
    buy_product("Gaming-Console", 500, user)

if __name__ == "__main__":
    main()

--- Buying Gaming-Console ---
📦 Checking Stock: Gaming-Console
💳 Charged $500 to card VISA-4242
🚚 Shipped Gaming-Console to NY, USA
✅ Purchase successful!


#### Key Differences

| Feature        | Classic OOP                                                    | Pythonic                                                      |
|----------------|----------------------------------------------------------------|----------------------------------------------------------------|
| **Structure**  | A wrapper class (`OrderFacade`) containing subsystem objects.  | A wrapper function (`buy_product`) or a package (`__init__.py`). |
| **Visibility** | Uses private fields to hide subsystems.                        | Uses naming conventions (`_function`) or `__all__` to hide internals. |
| **Usage**      | `facade.do_something()`                                        | `package.do_something()`                                      |


#### When to use which?

- **Java Way**: Use this when you need to maintain **state** inside the facade (e.g., if the Facade needs to cache the connection to the Inventory Database).
- **Pythonic Way**: Use this for stateless workflows. If your goal is just to simplify imports and function calls for the user, a simple "Wrapper Function" or exposing specific functions in `__init__`.py is the standard Python practice.